# Wildlife Camera EDA In R

R version of the first-pass exploratory analysis for the SDRPF wildlife camera dataset. This notebook looks at data quality, species composition, camera/preserve coverage, seasonality, daily activity patterns, and 30-minute independent events.

## Setup

If imports fail, install the R packages in the notebook kernel first:

```r
install.packages(c("tidyverse", "lubridate", "scales"))
```

In [ ]:
library(tidyverse)
library(lubridate)
library(scales)

theme_set(theme_minimal(base_size = 12))
options(dplyr.summarise.inform = FALSE)

DATA_PATH <- file.path("data", "Derek Master Sheet 2012-2018 - All Data.csv")

## Load And Clean

In [ ]:
season_map <- c(
  "1" = "Winter", "2" = "Winter", "3" = "Spring", "4" = "Spring",
  "5" = "Spring", "6" = "Summer", "7" = "Summer", "8" = "Summer",
  "9" = "Fall", "10" = "Fall", "11" = "Fall", "12" = "Winter"
)

raw <- read_csv(DATA_PATH, na = c("", "NA", "N/A"), show_col_types = FALSE)

df <- raw %>%
  rename_with(str_trim) %>%
  mutate(across(where(is.character), str_squish)) %>%
  mutate(
    timestamp_raw = timestamp,
    timestamp = parse_date_time(timestamp_raw, orders = c("ymd HMS", "ymd HM"), quiet = TRUE),
    year = as.integer(year),
    month = as.integer(month),
    day = as.integer(day),
    date = as.Date(timestamp),
    hour = hour(timestamp),
    weekday = wday(timestamp, label = TRUE, abbr = FALSE),
    week = floor_date(timestamp, unit = "week"),
    year_month = floor_date(timestamp, unit = "month"),
    species_label = if_else(is.na(common_name) | common_name == "", "Unknown", common_name),
    season = unname(season_map[as.character(month)]),
    day_part = case_when(
      is.na(hour) ~ NA_character_,
      between(hour, 5, 8) ~ "Dawn",
      between(hour, 9, 16) ~ "Day",
      between(hour, 17, 20) ~ "Dusk",
      TRUE ~ "Night"
    )
  )

glimpse(df)
head(df)

In [ ]:
overview <- tibble(
  metric = c(
    "rows", "columns", "date_min", "date_max", "preserves", "cameras",
    "deployments", "common_names", "exact_duplicate_rows", "unparsed_timestamps"
  ),
  value = c(
    nrow(df), ncol(df), as.character(min(df$timestamp, na.rm = TRUE)),
    as.character(max(df$timestamp, na.rm = TRUE)), n_distinct(df$preserve, na.rm = TRUE),
    n_distinct(df$camera, na.rm = TRUE), n_distinct(df$deployment_id, na.rm = TRUE),
    n_distinct(df$species_label, na.rm = TRUE), sum(duplicated(df)), sum(is.na(df$timestamp))
  )
)

overview

## Data Quality Checks

In [ ]:
missing <- df %>%
  summarise(across(everything(), ~ sum(is.na(.x)))) %>%
  pivot_longer(everything(), names_to = "column", values_to = "missing") %>%
  mutate(pct = missing / nrow(df)) %>%
  arrange(desc(missing))

missing

In [ ]:
duplicate_examples <- df %>%
  filter(duplicated(.) | duplicated(., fromLast = TRUE)) %>%
  arrange(timestamp, camera, species_label)

cat("Exact duplicate rows:", comma(sum(duplicated(df))), "\n")
head(duplicate_examples, 20)

In [ ]:
records_by_year <- df %>%
  count(year, name = "records") %>%
  arrange(year)

records_by_year

## Independent Events

Camera trap datasets often contain bursts of photos from the same animal visit. The helper below creates 30-minute independent events by `camera` and `species_label`. Change `EVENT_GAP_MINUTES` if your project uses a different standard.

In [ ]:
EVENT_GAP_MINUTES <- 30

events <- df %>%
  filter(!is.na(timestamp)) %>%
  arrange(camera, species_label, timestamp) %>%
  group_by(camera, species_label) %>%
  mutate(
    gap_minutes = as.numeric(difftime(timestamp, lag(timestamp), units = "mins")),
    new_event = is.na(gap_minutes) | gap_minutes > EVENT_GAP_MINUTES,
    event_id = cumsum(new_event)
  ) %>%
  ungroup()

event_summary <- events %>%
  group_by(camera, species_label, event_id) %>%
  summarise(
    preserve = first(preserve),
    deployment_id = first(deployment_id),
    latitude = first(latitude),
    longitude = first(longitude),
    event_start = min(timestamp),
    event_end = max(timestamp),
    photos = n(),
    .groups = "drop"
  ) %>%
  mutate(
    duration_minutes = as.numeric(difftime(event_end, event_start, units = "mins")),
    year = year(event_start),
    month = month(event_start),
    hour = hour(event_start),
    season = unname(season_map[as.character(month)])
  )

cat("Raw detections with timestamps:", comma(nrow(events)), "\n")
cat("Independent", EVENT_GAP_MINUTES, "minute events:", comma(nrow(event_summary)), "\n")
head(event_summary)

## Species Composition

In [ ]:
top_species <- df %>% count(species_label, sort = TRUE) %>% slice_head(n = 20)
top_event_species <- event_summary %>% count(species_label, sort = TRUE) %>% slice_head(n = 20)

ggplot(top_species, aes(x = n, y = reorder(species_label, n))) +
  geom_col(fill = "#4477AA") +
  labs(title = "Top Common Names: Raw Detections", x = "Detections", y = NULL)

In [ ]:
ggplot(top_event_species, aes(x = n, y = reorder(species_label, n))) +
  geom_col(fill = "#228833") +
  labs(title = paste0("Top Common Names: ", EVENT_GAP_MINUTES, "-Minute Events"), x = "Events", y = NULL)

In [ ]:
species_table <- df %>%
  group_by(species_label) %>%
  summarise(
    detections = n(),
    preserves = n_distinct(preserve, na.rm = TRUE),
    cameras = n_distinct(camera, na.rm = TRUE),
    first_seen = min(timestamp, na.rm = TRUE),
    last_seen = max(timestamp, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  left_join(event_summary %>% count(species_label, name = "events_30m"), by = "species_label") %>%
  mutate(events_30m = replace_na(events_30m, 0L)) %>%
  arrange(desc(detections))

head(species_table, 25)

## Camera And Preserve Coverage

In [ ]:
preserve_counts <- df %>% count(preserve, sort = TRUE)

ggplot(preserve_counts, aes(x = reorder(preserve, -n), y = n)) +
  geom_col(fill = "#66CCEE") +
  labs(title = "Raw Detections By Preserve", x = "Preserve", y = "Detections")

In [ ]:
camera_counts <- df %>% count(camera, sort = TRUE) %>% slice_head(n = 20)

ggplot(camera_counts, aes(x = n, y = reorder(camera, n))) +
  geom_col(fill = "#CCBB44") +
  labs(title = "Top Cameras By Raw Detections", x = "Detections", y = "Camera")

In [ ]:
camera_locations <- df %>%
  group_by(preserve, camera, latitude, longitude) %>%
  summarise(detections = n(), species = n_distinct(species_label), .groups = "drop")

ggplot(camera_locations, aes(x = longitude, y = latitude, color = preserve, size = detections)) +
  geom_point(alpha = 0.8) +
  scale_size_continuous(range = c(2, 12), labels = comma) +
  labs(title = "Camera Locations Sized By Detections", x = "Longitude", y = "Latitude", size = "Detections")

## Trends Over Time

In [ ]:
monthly <- df %>%
  filter(!is.na(year_month)) %>%
  count(year_month, name = "detections")

ggplot(monthly, aes(x = year_month, y = detections)) +
  geom_line(color = "#4477AA", linewidth = 0.8) +
  labs(title = "Monthly Raw Detections", x = NULL, y = "Detections")

In [ ]:
top10 <- df %>% count(species_label, sort = TRUE) %>% slice_head(n = 10) %>% pull(species_label)

species_year <- df %>%
  filter(species_label %in% top10) %>%
  count(year, species_label, name = "detections")

ggplot(species_year, aes(x = year, y = detections, color = species_label)) +
  geom_line(linewidth = 0.8) +
  geom_point(size = 1.8) +
  labs(title = "Top Species Detections By Year", x = "Year", y = "Detections", color = "Common name")

In [ ]:
monthly_species <- df %>%
  filter(species_label %in% top10, !is.na(month)) %>%
  count(species_label, month, name = "detections")

ggplot(monthly_species, aes(x = factor(month, levels = 1:12), y = species_label, fill = detections)) +
  geom_tile(color = "white") +
  scale_fill_viridis_c(labels = comma) +
  labs(title = "Seasonality Heatmap For Top Species", x = "Month", y = NULL, fill = "Detections")

## Daily Activity Patterns

In [ ]:
hourly_events <- event_summary %>%
  filter(species_label %in% top10, !is.na(hour)) %>%
  count(species_label, hour, name = "events")

ggplot(hourly_events, aes(x = factor(hour, levels = 0:23), y = species_label, fill = events)) +
  geom_tile(color = "white") +
  scale_fill_viridis_c(labels = comma) +
  labs(title = paste0("Hourly Activity For Top Species (", EVENT_GAP_MINUTES, "-Minute Events)"), x = "Hour Of Day", y = NULL, fill = "Events")

In [ ]:
day_part_order <- c("Dawn", "Day", "Dusk", "Night")

day_part <- df %>%
  filter(species_label %in% top10, !is.na(day_part)) %>%
  count(species_label, day_part, name = "detections") %>%
  group_by(species_label) %>%
  mutate(share = detections / sum(detections)) %>%
  ungroup() %>%
  mutate(day_part = factor(day_part, levels = day_part_order))

ggplot(day_part, aes(x = share, y = species_label, fill = day_part)) +
  geom_col() +
  scale_x_continuous(labels = percent) +
  labs(title = "Share Of Detections By Day Part", x = "Share", y = NULL, fill = "Day part")

## Starter Questions To Pursue

- Which species are widespread across preserves versus concentrated at a few cameras?
- Do raw detection counts tell a different story than 30-minute independent events?
- Are there years or months with sharp changes that reflect camera effort rather than wildlife activity?
- Which species are mainly nocturnal, diurnal, or crepuscular?
- Which taxonomy gaps should be reviewed manually, especially common names such as family-level or generic labels?

In [ ]:
# Optional exports after reviewing the cleaning choices above.
# dir.create("outputs", showWarnings = FALSE)
# write_csv(df, file.path("outputs", "wildlife_camera_cleaned.csv"))
# write_csv(event_summary, file.path("outputs", "wildlife_camera_events_30m.csv"))